# Módulo 07 · Lista de Exercícios

> **Manual de Estudos Interativo** · Trilha Engenharia de Software & Dados
> Projeto transversal: **Atlas / Aurora Comércio**

Este módulo mudou o seu papel duas vezes: de servidor para **cliente**, e de quem escreve código para quem **garante** que ele continua funcionando.

## Como usar

| | |
|---|---|
| 🟢 **Aquecimento** | 1–10 · uma ideia por exercício |
| 🟡 **Construção** | 11–26 · combinam conceitos |
| 🔴 **Integração** | 27–40 · perto de produção |
| 🏗️ **Projeto** | Atlas conectado ao mundo |

**Regras de casa:**

1. Preveja antes de rodar. Escreva num comentário o que espera.
2. Todo exercício marcado 🔴 tem uma armadilha. Encontre-a antes de ler a dica.
3. Os exercícios de teste (31–40) valem por dois: eles são o que sobra quando você esquecer o resto.

## ⚙️ Preparação

**Execute a célula abaixo antes de tudo.**

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Preparação — Lista do Módulo 07
# ═══════════════════════════════════════════════════════════════
import json
import shutil
import socket
import subprocess
import sys
import threading
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")


def _garantir(pacote, importar=None):
    nome = importar or pacote
    try:
        __import__(nome)
        return True
    except ImportError:
        print(f"  instalando {pacote}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pacote, "--quiet"],
                       check=False, capture_output=True)
        try:
            __import__(nome)
            return True
        except ImportError:
            print(f"  ⚠️ {pacote} indisponível")
            return False


for _p, _m in [("fastapi", "fastapi"), ("httpx", "httpx"),
               ("uvicorn[standard]", "uvicorn"), ("pytest", "pytest"),
               ("respx", "respx"), ("fakeredis", "fakeredis"),
               ("sqlalchemy", "sqlalchemy"), ("python-multipart", "multipart")]:
    _garantir(_p, _m)

import httpx
import uvicorn
from fastapi import FastAPI
from fastapi.testclient import TestClient

print(f"✅ httpx {httpx.__version__}")

BASE = Path("lista_07").resolve()
if BASE.exists():
    shutil.rmtree(BASE)
(BASE / "tests").mkdir(parents=True)
sys.path = [str(BASE)] + [p for p in sys.path if p != str(BASE)]
print(f"📁 {BASE}")


# ═══════════════════════════════════════════════════════════════
#  Servidor real numa thread
# ═══════════════════════════════════════════════════════════════

def _porta_livre() -> int:
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


class Servico:
    def __init__(self, app: FastAPI, nome: str = "servico"):
        self.app, self.nome = app, nome
        self.porta = _porta_livre()
        self.url = f"http://127.0.0.1:{self.porta}"
        self._servidor = self._thread = None

    def iniciar(self, timeout: float = 15.0) -> "Servico":
        config = uvicorn.Config(self.app, host="127.0.0.1", port=self.porta,
                                log_level="critical", access_log=False)
        self._servidor = uvicorn.Server(config)
        self._thread = threading.Thread(target=self._servidor.run, daemon=True)
        self._thread.start()
        limite = time.monotonic() + timeout
        while time.monotonic() < limite:
            if self._servidor.started:
                print(f"🟢 {self.nome} no ar em {self.url}")
                return self
            time.sleep(0.05)
        raise RuntimeError(f"{self.nome} não subiu")

    def parar(self) -> None:
        if self._servidor is not None:
            self._servidor.should_exit = True
            self._thread.join(timeout=10)
            print(f"⚫ {self.nome} desligado")

    def __enter__(self):
        return self.iniciar().url

    def __exit__(self, *_):
        self.parar()


# ═══════════════════════════════════════════════════════════════
#  Auxiliares
# ═══════════════════════════════════════════════════════════════

def mostrar(resposta, rotulo: str = "", corpo: bool = True, linhas: int = 10):
    cor = {2: "✅", 3: "↪️", 4: "⚠️", 5: "🔴"}.get(resposta.status_code // 100, "  ")
    alvo = rotulo or f"{resposta.request.method} {resposta.request.url.path}"
    print(f"{cor} {alvo:<44} → {resposta.status_code}")
    if corpo:
        try:
            texto = json.dumps(resposta.json(), ensure_ascii=False, indent=2)
            partes = texto.splitlines()
            for linha in partes[:linhas]:
                print(f"   {linha}")
            if len(partes) > linhas:
                print(f"   ... (+{len(partes) - linhas} linhas)")
        except Exception:
            if resposta.text.strip():
                print(f"   {resposta.text[:180]}")
    return resposta


def pytest_(*args, cwd: Path = BASE, resumo: bool = True, linhas: int = 40) -> int:
    """Roda o pytest de verdade, num subprocesso."""
    comando = [sys.executable, "-m", "pytest", "-p", "no:cacheprovider", *args]
    processo = subprocess.run(comando, cwd=cwd, capture_output=True, text=True)
    saida = (processo.stdout + processo.stderr).splitlines()
    if resumo and len(saida) > linhas:
        for linha in saida[:linhas // 2]:
            print(linha)
        print(f"   … (+{len(saida) - linhas} linhas)")
        for linha in saida[-(linhas // 2):]:
            print(linha)
    else:
        for linha in saida:
            print(linha)
    print(f"\n[saída do processo: {processo.returncode}]")
    return processo.returncode


print("✅ `Servico`, `mostrar()` e `pytest_()` prontos")

---

# 🟢 Aquecimento (1–10)

### 1 · `Client` vs chamada solta

Meça 50 requisições com `httpx.get()` solto e com um `httpx.Client`. Explique a diferença num comentário — e diga por que ela seria **muito maior** contra um servidor na internet.

In [ ]:
# 1

### 2 · 🔴 Timeout

Crie um cliente sem timeout e um com `timeout=1.0`. Aponte os dois para um endpoint que dorme 3 segundos.

Antes de rodar, escreva o que espera de cada um.

In [ ]:
# 2

### 3 · Os quatro timeouts

Explique num comentário, com uma frase cada, o que `connect`, `read`, `write` e `pool` medem — e qual deles denuncia um problema **seu**, não do servidor alheio.

In [ ]:
# 3

### 4 · Erro de resposta vs erro de rede

Provoque um `HTTPStatusError` e um `ConnectError`. Mostre que no primeiro você ainda tem `erro.response` e no segundo não existe resposta nenhuma.

In [ ]:
# 4

### 5 · `raise_for_status`

Mostre que o httpx devolve um `404` **sem levantar exceção**. Depois use `raise_for_status()` e explique por que esse é o padrão da biblioteca.

In [ ]:
# 5

### 6 · Paginação por offset

Escreva um gerador que percorra todas as páginas de um endpoint paginado e devolva item a item.

In [ ]:
# 6

### 7 · `Retry-After`

Faça 6 chamadas a um endpoint limitado a 3 por janela. Ao receber `429`, leia o cabeçalho `Retry-After` e espere exatamente o que ele pede.

In [ ]:
# 7

### 8 · Primeiro teste

Escreva `app/desconto.py` com `calcular_desconto(valor, cupom)` e três testes: cupom válido, cupom inválido e valor zero.

In [ ]:
# 8

### 9 · `parametrize`

Converta os três testes do exercício 8 num único `parametrize` com `id` legível para cada caso.

In [ ]:
# 9

### 10 · `pytest.raises`

Faça `calcular_desconto` levantar `ValueError` para valor negativo e teste com `pytest.raises(..., match=...)`.

In [ ]:
# 10

---

# 🟡 Construção (11–26)

### 11 · Cliente com token

Implemente uma classe cliente que obtenha um token e o renove **antes** de expirar, com margem. Prove a renovação contando as chamadas de autenticação.

In [ ]:
# 11

### 12 · Retry com backoff

Implemente retry com espera exponencial. Imprima cada tentativa e o tempo de espera.

In [ ]:
# 12

### 13 · 🔴 Jitter

Simule 50 clientes repetindo ao mesmo tempo, com e sem jitter. Mostre num histograma simples (contagem por faixa de 100 ms) que sem jitter todos voltam juntos.

In [ ]:
# 13

### 14 · A tabela do retry

Escreva uma função `deve_repetir(erro_ou_status, metodo)` que devolva `True`, `False` ou `"depende"`. Teste-a com os 15 casos da aula.

In [ ]:
# 14

### 15 · 🔴 `Idempotency-Key` no cliente

Implemente: gere a chave **uma vez por operação lógica** e reutilize-a em todas as tentativas daquela operação. Prove que duas tentativas mandam a mesma chave.

In [ ]:
# 15

### 16 · Disjuntor

Implemente com os três estados explícitos e demonstre a transição `aberto → meio-aberto → fechado`.

In [ ]:
# 16

### 17 · Cursor vs offset

Implemente as duas paginações contra o mesmo conjunto e explique num comentário qual você usaria para sincronizar 4.000 entregas de madrugada.

In [ ]:
# 17

### 18 · 🔴 Webhook: as três verificações

Implemente o receptor com assinatura HMAC, janela de timestamp e idempotência. Teste os três ataques.

In [ ]:
# 18

### 19 · `compare_digest`

Explique num comentário por que `==` é inseguro para comparar assinaturas. Se conseguir, meça a diferença.

In [ ]:
# 19

### 20 · Emissor de webhook

Implemente o outro lado: envia, espera confirmação, e reenvia com backoff se não receber.

In [ ]:
# 20

### 21 · 🔴 Upload seguro

Valide magic bytes, tamanho em streaming e nome do arquivo. Tente burlar cada uma das três defesas.

In [ ]:
# 21

### 22 · Streaming

Gere um CSV de 500 mil linhas com `StreamingResponse`. Prove que a memória do processo não cresce proporcionalmente.

In [ ]:
# 22

### 23 · Cache-aside

Implemente com `fakeredis`, com chaves versionadas e TTL. Meça os acertos.

In [ ]:
# 23

### 24 · Invalidação

Adicione invalidação em **todos** os caminhos de escrita. Escreva um teste que falhe se algum caminho for esquecido.

In [ ]:
# 24

### 25 · WebSocket

Implemente um painel que receba o estado inicial e depois as atualizações empurradas por uma rota HTTP.

In [ ]:
# 25

### 26 · Fixtures

Crie um `conftest.py` com fixtures `motor`, `sessao`, `cliente` e `catalogo`. Use as quatro em testes diferentes.

In [ ]:
# 26

---

# 🔴 Integração (27–40)

### 27 · Cliente resiliente completo

Junte timeouts, retry com jitter, disjuntor, `Retry-After` e paginação num único cliente. Exponha métricas (`chamadas`, `repeticoes`, `falhas`).

In [ ]:
# 27

### 28 · Degradação

Quando o disjuntor abrir, a sua API não deve devolver `500`. Implemente uma resposta degradada (frete estimado, com aviso) e justifique a escolha.

In [ ]:
# 28

### 29 · 🔴 Sincronização retomável

Sincronize 137 entregas. Se o processo morrer na página 2, a próxima execução deve continuar de onde parou — sem duplicar nem pular.

In [ ]:
# 29

### 30 · Limitador do lado cliente

Implemente um limitador que nunca ultrapasse N requisições por segundo — e portanto nunca receba um `429`.

In [ ]:
# 30

### 31 · 🔴 Isolamento de testes

Crie a fixture de banco isolado e escreva o teste que **prova** o isolamento. Depois quebre-o de propósito e mostre a falha.

In [ ]:
# 31

### 32 · `dependency_overrides`

Substitua o banco real por um de teste. Prove que os dados de teste não vazam para o banco de desenvolvimento.

In [ ]:
# 32

### 33 · 🔴 `StaticPool`

Monte a fixture **sem** `poolclass=StaticPool` e mostre o `no such table`. Depois corrija e explique a causa.

In [ ]:
# 33

### 34 · Testes de segurança

Escreva testes que verifiquem: nenhum esquema de resposta expõe `custo`, `limite` fora da faixa é recusado, e SKU fora do padrão é recusado.

In [ ]:
# 34

### 35 · `respx`

Teste o cliente da transportadora: sucesso, retry em `503`, ausência de retry em `422`, e erro de rede. Verifique `call_count` em cada caso.

In [ ]:
# 35

### 36 · Verificar o que foi enviado

Com `respx`, verifique o corpo, os cabeçalhos e a URL de cada requisição — não só a resposta.

In [ ]:
# 36

### 37 · Testar o que NÃO acontece

Escreva três testes que verifiquem comportamento **ausente**: não repetiu, não chamou, não gravou.

In [ ]:
# 37

### 38 · Cobertura enganosa

Escreva um teste que cubra 100% de um módulo sem verificar nada. Mostre o relatório e explique o que ele não diz.

In [ ]:
# 38

### 39 · Marcadores

Configure `lento`, `integracao` e `seguranca` com `--strict-markers`. Rode cada subconjunto e mostre um marcador com erro de digitação virando erro.

In [ ]:
# 39

### 40 · 🔴 Teste de regressão

Escolha um bug que você já cometeu neste manual. Escreva o teste que falha por causa dele, corrija, e veja o teste passar.

In [ ]:
# 40

---

# 🏗️ Projeto — Atlas conectado ao mundo

## O contexto

A Atlas API v1 está no ar (M06). Agora ela precisa **conversar**:

| Quem | O que precisa |
|------|---------------|
| Transportadora Veloz | cotação de frete e rastreamento |
| Gateway de pagamento | avisar quando o boleto é pago |
| Tela de estoque | não derrubar o banco com 900 consultas/minuto |
| **Você** | mexer no código sem medo |

## O que entregar

```
projeto_Atlas/
├── src/atlas/integracoes/
│   ├── __init__.py
│   ├── cliente_http.py       ← cliente base resiliente
│   ├── transportadora.py     ← Veloz: cotação, rastreio
│   ├── gateway.py            ← pagamento: cobrança
│   └── cache.py              ← cache-aside sobre Redis
├── src/atlas/api/rotas/
│   └── webhooks.py           ← recepção validada
├── tests/
│   ├── conftest.py
│   ├── test_frete.py
│   ├── test_api_produtos.py
│   ├── test_api_pedidos.py
│   ├── test_seguranca.py
│   ├── test_webhooks.py
│   └── test_integracoes.py   ← com respx
├── docs/INTEGRACOES.md
└── pyproject.toml            ← marcadores e cobertura
```

## Requisitos obrigatórios

| # | Requisito | Pronto quando |
|---|-----------|---------------|
| 1 | Todo cliente HTTP tem timeout | `grep -r "httpx.Client" src/` — nenhum sem `timeout` |
| 2 | Retry só nos status certos | `422` não é repetido |
| 3 | `POST` só repete com `Idempotency-Key` | teste prova |
| 4 | Disjuntor com degradação | API não devolve `500` quando o parceiro cai |
| 5 | 🔴 Webhook com as 3 verificações | assinatura, timestamp, idempotência |
| 6 | Idempotência sobrevive ao restart | Redis ou tabela, não `set` em memória |
| 7 | Cache com chave versionada | e invalidação em toda escrita |
| 8 | Suíte de testes rodando | `pytest` sai com código 0 |
| 9 | 🔴 Testes isolados | ordem não importa; existe teste que prova |
| 10 | Integrações testadas com `respx` | nenhum teste toca a internet |
| 11 | `docs/INTEGRACOES.md` | com as decisões, não só a descrição |

> 📋 **O roteiro passo a passo está em `projeto_Atlas/ROTEIRO_M07.md`.**
>
> 🔴 **O esqueleto tem apenas assinaturas e `# TODO`.** O código é seu.

## 🧪 Bateria de aceitação

Aponte para a sua implementação e faça tudo passar.

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Aponte para a SUA suíte
# ═══════════════════════════════════════════════════════════════
#
#   PROJETO = Path(r"C:\...\Roadmap\projeto_Atlas")
#
# Enquanto não apontar, as células avisam e não quebram o notebook.

PROJETO = None          # ← troque pelo caminho do seu projeto_Atlas

RESULTADOS = []


def checar(nome: str, condicao: bool, detalhe: str = "") -> bool:
    RESULTADOS.append((nome, bool(condicao)))
    print(f"   {'✅' if condicao else '🔴'} {nome}{('  — ' + detalhe) if detalhe else ''}")
    return bool(condicao)


def placar():
    passou = sum(1 for _, ok in RESULTADOS if ok)
    print(f"\n{'═' * 54}\n  {passou}/{len(RESULTADOS)} verificações passaram")
    if RESULTADOS and passou == len(RESULTADOS):
        print("  🎉 Atlas integrado e testado.")
    else:
        for nome, ok in RESULTADOS:
            if not ok:
                print(f"  🔴 pendente: {nome}")
    print("═" * 54)


if PROJETO is None:
    print("⏸️  defina `PROJETO` acima para rodar a bateria.")
else:
    print(f"▶️  auditando {PROJETO}")

In [ ]:
# ── Bateria 1: a suíte de testes existe e passa ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    pasta_testes = PROJETO / "tests"
    checar("existe a pasta tests/", pasta_testes.is_dir())
    checar("existe conftest.py", (pasta_testes / "conftest.py").exists())

    arquivos = sorted(pasta_testes.glob("test_*.py")) if pasta_testes.is_dir() else []
    checar("há ao menos 4 arquivos de teste", len(arquivos) >= 4,
           f"{[a.name for a in arquivos]}")

    if arquivos:
        print("\n   ── rodando a sua suíte ──")
        codigo = pytest_("-q", "--tb=line", cwd=PROJETO, resumo=True, linhas=24)
        checar("🔴 pytest sai com código 0", codigo == 0, f"código {codigo}")

    placar()

In [ ]:
# ── Bateria 2: 🔴 cada teste passa SOZINHO ──
#
# 🎯 Este é o detector de verdade para dependência de ordem.
#
#    Rodar a suíte duas vezes não basta: um estado que vive num módulo
#    Python é recriado a cada processo, então o problema não aparece.
#
#    O que denuncia é rodar cada teste ISOLADAMENTE. Um teste que só
#    passa depois de outro ter rodado vai falhar aqui — e é exatamente
#    esse teste que um dia vai quebrar o CI sem motivo aparente.
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    coleta = subprocess.run(
        [sys.executable, "-m", "pytest", "-p", "no:cacheprovider",
         "--collect-only", "-q", "--no-header"],
        cwd=PROJETO, capture_output=True, text=True)

    ids = [linha.strip() for linha in coleta.stdout.splitlines()
           if "::" in linha and not linha.startswith(" ")]
    checar("a suíte é coletável", bool(ids), f"{len(ids)} testes")

    TETO = 40           # ⏱️ um subprocesso por teste; limitamos o tempo
    amostra = ids[:TETO]
    if len(ids) > TETO:
        print(f"   (auditando os {TETO} primeiros de {len(ids)})")

    sozinhos_falham = []
    for id_teste in amostra:
        r = subprocess.run(
            [sys.executable, "-m", "pytest", "-p", "no:cacheprovider",
             "-q", "--tb=no", id_teste],
            cwd=PROJETO, capture_output=True, text=True)
        if r.returncode != 0:
            sozinhos_falham.append(id_teste)

    checar("a suíte inteira passa",
           pytest_("-q", "--tb=no", cwd=PROJETO, resumo=True, linhas=6) == 0)
    checar("🔴 todo teste passa TAMBÉM sozinho", not sozinhos_falham,
           f"{len(sozinhos_falham)} dependem de outro: {sozinhos_falham[:3]}")

    placar()

In [ ]:
# ── Bateria 3: 🔴 nenhum cliente HTTP sem timeout ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    import re

    fontes = list((PROJETO / "src").rglob("*.py")) if (PROJETO / "src").is_dir() else []
    sem_timeout = []
    for arquivo in fontes:
        texto = arquivo.read_text(encoding="utf-8", errors="ignore")
        # procura httpx.Client(...) / httpx.AsyncClient(...) sem `timeout`
        for trecho in re.findall(r"httpx\.(?:Async)?Client\((?:[^()]|\([^()]*\))*\)",
                                 texto, re.DOTALL):
            if "timeout" not in trecho:
                sem_timeout.append(f"{arquivo.name}: {trecho[:60]}…")

    checar("🔴 todo httpx.Client declara timeout", not sem_timeout,
           str(sem_timeout[:2]))

    # nenhum segredo no código
    suspeitas = []
    for arquivo in fontes:
        for i, linha in enumerate(
                arquivo.read_text(encoding="utf-8", errors="ignore").splitlines(), 1):
            nua = linha.split("#")[0]
            if re.search(r'(secret|senha|password|token|api_key)\s*=\s*["\'][^"\']{8,}',
                         nua, re.IGNORECASE):
                suspeitas.append(f"{arquivo.name}:{i}")
    checar("🔴 nenhum segredo literal no código", not suspeitas, str(suspeitas[:3]))

    placar()

In [ ]:
# ── Bateria 4: 🔒 os testes cobrem o que importa ──
RESULTADOS.clear()

if PROJETO is None:
    print("⏸️  defina `PROJETO`")
else:
    todos = "\n".join(
        a.read_text(encoding="utf-8", errors="ignore")
        for a in (PROJETO / "tests").rglob("test_*.py")
    ) if (PROJETO / "tests").is_dir() else ""

    temas = {
        "custo não vaza":            ["custo"],
        "autorização (401/403)":     ["401", "403"],
        "conflito (409)":            ["409"],
        "validação (422)":           ["422"],
        "integração com respx":      ["respx"],
        "webhook / assinatura":      ["webhook", "assinatura", "hmac"],
        "atomicidade do estoque":    ["estoque"],
        "idempotência":              ["idempot", "repetid"],
    }
    for tema, palavras in temas.items():
        achou = any(p.lower() in todos.lower() for p in palavras)
        checar(f"há teste sobre {tema}", achou)

    placar()

> 🎯 **A bateria 2 é a que mais gente reprova — e a que eu mesmo errei ao escrever esta lista.**
>
> A primeira versão dela rodava a suíte **duas vezes** e considerava isso uma prova de isolamento. Não é: um estado guardado num módulo Python é recriado a cada processo, então a segunda execução passa exatamente como a primeira. O teste sujo passava despercebido.
>
> **O que denuncia de verdade é rodar cada teste sozinho.** Um teste que só passa depois de outro ter rodado falha isolado — e é justamente ele que um dia vai deixar o CI vermelho sem que ninguém tenha mexido em nada relacionado.
>
> 💭 A lição vale além dos testes: **uma verificação que nunca reprova ninguém provavelmente não está verificando nada.** Antes de confiar num teste, quebre o código de propósito e veja se ele reclama.

---

## 🎓 Autoavaliação

| # | Consigo… | ✅ |
|---|----------|---|
| 1 | Explicar por que um `Client` é melhor que chamadas soltas | |
| 2 | Dizer o que cada um dos quatro timeouts mede | |
| 3 | Explicar por que `timeout=None` é perigoso | |
| 4 | Distinguir "não houve resposta" de "a resposta foi ruim" | |
| 5 | Listar os status que se repete e os que não | |
| 6 | Explicar por que o `ReadTimeout` é ambíguo | |
| 7 | Explicar o que o `Idempotency-Key` resolve | |
| 8 | Justificar o jitter no backoff | |
| 9 | Dizer o que o disjuntor resolve que o retry não resolve | |
| 10 | Explicar quando cursor é melhor que offset | |
| 11 | Listar as três verificações de um webhook | |
| 12 | Explicar por que se lê o corpo **cru** no webhook | |
| 13 | Explicar por que `==` não serve para comparar assinatura | |
| 14 | Listar as três defesas de um upload | |
| 15 | Explicar por que `Content-Type` não é confiável | |
| 16 | Dizer quando `BackgroundTasks` **não** basta | |
| 17 | Desenhar uma chave de cache e justificar cada parte | |
| 18 | Explicar o estouro de cache e três defesas | |
| 19 | Dizer quando polling basta e WebSocket é exagero | |
| 20 | Escrever uma fixture de banco isolado de memória | |
| 21 | Explicar por que `StaticPool` é necessário | |
| 22 | Usar `dependency_overrides` sem consultar | |
| 23 | Explicar por que escopo amplo com objeto mutável é armadilha | |
| 24 | Testar uma API externa sem internet | |
| 25 | Escrever um teste que verifica comportamento **ausente** | |
| 26 | Explicar por que cobertura alta não significa testado | |
| 27 | Escrever um teste de regressão a partir de um bug real | |

**Menos de 21?** Volte às aulas correspondentes antes do Módulo 08.

---

### ➡️ Próximo módulo

**Módulo 08 — Docker.** *"Configurar a máquina de um dev leva 2 dias."*

O Atlas hoje precisa de Python, PostgreSQL, MongoDB, Redis e um punhado de variáveis de ambiente. No próximo módulo, tudo isso vira um comando.